# Archive Legacy Artifacts: Inventory

This notebook mirrors `inventory_legacy_artifacts.py` and keeps the same top-to-bottom order as the production script. The Makefile remains the source of truth for production dependencies; use this notebook when you want to inspect intermediate objects, change a small parameter temporarily, or rerun one stage at a time.

Run the notebook from this task's `code/` folder when possible. If the kernel starts from the repository root or the task folder, the setup cell below will try to find the right `code/` directory.

Running output-writing cells can overwrite files in `../output/`, just like running the script through `make`.


## Original Script Header

```text
Purpose:
    Inventory legacy artifacts that are not promoted into the production task graph.

Inputs:
    ../../../legacy/mmb_upgraded/

Outputs:
    ../output/artifact_inventory.txt

Run:
    make from tasks/legacy/artifact_inventory/code/
```

In [ ]:
from pathlib import Path
from collections import Counter

import pandas as pd


code_dir = Path.cwd().resolve()
if code_dir.name != "code":
    task_code_candidate = code_dir / "code"
    repo_code_candidate = Path("tasks/legacy/artifact_inventory/code").resolve()
    if task_code_candidate.exists():
        code_dir = task_code_candidate
    elif repo_code_candidate.exists():
        code_dir = repo_code_candidate
    else:
        raise RuntimeError("Start this notebook from the task code folder, the task folder, or the repository root.")
task_dir = code_dir.parent
repo_dir = task_dir.parents[2]
legacy_dir = repo_dir / "legacy" / "mmb_upgraded"
output_dir = task_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print

## Inventory legacy tree

In [ ]:
rows = []
extension_counts = Counter()
extension_sizes = Counter()

for path in sorted(legacy_dir.rglob("*")):
    if ".git" in path.parts or not path.is_file():
        continue
    relative = path.relative_to(repo_dir)
    extension = path.suffix.lower() if path.suffix else "[no extension]"
    size_bytes = path.stat().st_size
    extension_counts[extension] += 1
    extension_sizes[extension] += size_bytes

    first_part = path.relative_to(legacy_dir).parts[0]
    if path.name == "Model_Characteristics_corrections.xlsx":
        disposition = "promoted as main model-characteristics input through tasks/data/import_mmb_legacy_data"
    elif path.name.startswith("Model_Characteristics"):
        disposition = "legacy model-characteristics reference; not the main feed-in input"
    elif str(relative).startswith("legacy/mmb_upgraded/data/raw"):
        disposition = "promoted as source input through tasks/data/import_mmb_legacy_data"
    elif str(relative).startswith("legacy/mmb_upgraded/data/derived"):
        disposition = "superseded by tasks/data/build_mmb_analysis_dataset outputs"
    elif str(relative).startswith("legacy/mmb_upgraded/output/cloud_graphs"):
        disposition = "superseded by tasks/graphs/generate_mmb_cloud_graphs outputs"
    elif first_part in ["documentation", "drafts", "outlier_emails"]:
        disposition = "reference archive; keep in legacy until cited by a specific task"
    elif first_part in ["code", "work_code_legacy", "misc", "calc-sacrifice-ratio"]:
        disposition = "legacy code archive; promote only when needed by a named task"
    elif first_part in ["output", "bob", "cloud_graphs", "outlier_charts"]:
        disposition = "legacy generated output; preserved exactly by tasks/legacy/generated_output_archives"
    else:
        disposition = "legacy archive; inspect before promoting"

    rows.append(
        {
            "path": str(relative),
            "extension": extension,
            "size_bytes": size_bytes,
            "disposition": disposition,
        }
    )

In [ ]:
inventory_preview = pd.DataFrame(rows)
print(f"Inventory rows available so far: {inventory_preview.shape}")
display(inventory_preview.head(10))

## Write summary and full inventory

In [ ]:
inventory = pd.DataFrame(rows)
summary_rows = []
for extension, count in extension_counts.most_common():
    summary_rows.append(
        {
            "extension": extension,
            "files": count,
            "megabytes": round(extension_sizes[extension] / 1024 / 1024, 2),
        }
    )
summary = pd.DataFrame(summary_rows)

with open(output_dir / "artifact_inventory.txt", "w", encoding="utf-8") as f:
    f.write("Legacy artifact inventory\n")
    f.write("=========================\n\n")
    f.write(f"Legacy root: {legacy_dir.relative_to(repo_dir)}\n")
    f.write(f"Files inventoried outside nested .git folders: {len(inventory)}\n\n")
    f.write("Summary by extension\n")
    f.write("--------------------\n")
    f.write(summary.to_string(index=False))
    f.write("\n\n")
    f.write("Disposition rules\n")
    f.write("-----------------\n")
    f.write("Core raw data enter through the source-data import task.\n")
    f.write("Derived datasets and cloud graphs are rebuilt in production tasks.\n")
    f.write("Exact generated legacy artifacts are preserved by tasks/legacy/generated_output_archives for parity checks and reference.\n")
    f.write("Old drafts, exploratory notebooks, and one-off code remain archived until a named task needs them.\n\n")
    f.write("Full inventory\n")
    f.write("--------------\n")
    f.write(inventory.to_string(index=False))
    f.write("\n")

print(f"Wrote legacy inventory for {len(inventory)} files.")

In [ ]:
print(f"Full inventory rows x columns: {inventory.shape}")
print(f"Extension summary rows x columns: {summary.shape}")
display(summary.head(20))